In [2]:
%%writefile mandelbrot_cpu.cpp
#include <cstdint>
#include <iostream>
#include <fstream>
#include <vector>
#include <chrono>
#include <cstring>
#include <cstdlib>
#include <immintrin.h>

// Aligned allocator for AVX-512 (64-byte alignment)
template<typename T>
class AlignedAllocator {
public:
    using value_type = T;

    AlignedAllocator() = default;
    template<typename U> AlignedAllocator(const AlignedAllocator<U>&) {}

    T* allocate(size_t n) {
        size_t size = n * sizeof(T);
        size = (size + 63) & ~63;  // Round up to 64-byte boundary
        void* ptr = std::aligned_alloc(64, size);
        if (!ptr) throw std::bad_alloc();
        return static_cast<T*>(ptr);
    }

    void deallocate(T* p, size_t) { std::free(p); }
};

template<typename T>
using aligned_vector = std::vector<T, AlignedAllocator<T>>;

#ifndef __AVX512F__
#error "This source file is AVX-512-only. Compile with AVX-512 enabled (e.g. -mavx512f)."
#endif

void mandelbrot_cpu_scalar(
    uint32_t img_size, /* usually greater than 100 */
    uint32_t max_iters, /* usually greater than 100 */
    uint32_t *out /* buffer of 'img_size * img_size' integers */
  ) {
  for (uint32_t i = 0; i < img_size; i++) {
    for (uint32_t j = 0; j < img_size; j++) {
        float cx = (float(j) / float(img_size)) * 2.5f - 2.0f;
        float cy = (float(i) / float(img_size)) * 2.5f - 1.25f;

        float x2 = 0.0f;
        float y2 = 0.0f;
        float w = 0.0f;
        uint32_t iters = 0;
        while (x2 + y2 <= 4.0f && iters < max_iters) {
            float x = x2 - y2 + cx;
            float y = w - x2 - y2 + cy;
            x2 = x * x;
            y2 = y * y;
            w = (x + y) * (x + y);
            ++iters;
        }
        out[i * img_size + j] = iters;
    }
  }
}

// x86 AVX-512 vector implementation (512-bit vectors = 16 floats)
void mandelbrot_cpu_vector(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out
) {
    const float inv_img = 1.0f / float(img_size);
    const __m512 v_step = _mm512_set1_ps(2.5f * inv_img);
    const __m512 v_offx = _mm512_set1_ps(-2.0f);
    const __m512 v_four = _mm512_set1_ps(4.0f);
    const __m512i v_one = _mm512_set1_epi32(1);

    // [0..15]
    const __m512 v_idx = _mm512_setr_ps(
        0.f, 1.f, 2.f, 3.f, 4.f, 5.f, 6.f, 7.f,
        8.f, 9.f,10.f,11.f,12.f,13.f,14.f,15.f
    );

    for (uint32_t i = 0; i < img_size; ++i) {
        const float cy_s = (float(i) * inv_img) * 2.5f - 1.25f;
        const __m512 v_cy = _mm512_set1_ps(cy_s);

        for (uint32_t j = 0; j < img_size; j += 16) {
            const uint32_t remain = img_size - j;
            const __mmask16 lane_mask = (remain >= 16) ? (__mmask16)0xFFFF : (__mmask16)((1u << remain) - 1u);

            // cx = ((j + [0..15]) * step) + offx
            const __m512 v_jbase = _mm512_set1_ps((float)j);
            const __m512 v_j = _mm512_add_ps(v_jbase, v_idx);
            const __m512 v_cx = _mm512_add_ps(_mm512_mul_ps(v_j, v_step), v_offx);

            __m512  v_x2 = _mm512_setzero_ps();
            __m512  v_y2 = _mm512_setzero_ps();
            __m512  v_w  = _mm512_setzero_ps();
            __m512i v_it = _mm512_setzero_epi32();

            for (uint32_t iter = 0; iter < max_iters; ++iter) {
                // Check condition: x2 + y2 <= 4.0
                const __m512 v_sum = _mm512_add_ps(v_x2, v_y2);
                const __mmask16 m_in = _mm512_cmp_ps_mask(v_sum, v_four, _CMP_LE_OQ);

                // Early exit if all lanes escaped
                if (!m_in) break;

                // Calculate x and y with EXACT same order as scalar
                // x = x2 - y2 + cx
                const __m512 v_x = _mm512_add_ps(_mm512_sub_ps(v_x2, v_y2), v_cx);
                // y = w - x2 - y2 + cy (NOT w - (x2+y2) + cy due to float precision!)
                const __m512 v_y = _mm512_add_ps(_mm512_sub_ps(_mm512_sub_ps(v_w, v_x2), v_y2), v_cy);

                // Calculate new values
                const __m512 v_x2n = _mm512_mul_ps(v_x, v_x);
                const __m512 v_y2n = _mm512_mul_ps(v_y, v_y);
                const __m512 v_xy = _mm512_add_ps(v_x, v_y);
                const __m512 v_wn = _mm512_mul_ps(v_xy, v_xy);

                // Update state with mask to prevent NaN
                v_x2 = _mm512_mask_mov_ps(v_x2, m_in, v_x2n);
                v_y2 = _mm512_mask_mov_ps(v_y2, m_in, v_y2n);
                v_w = _mm512_mask_mov_ps(v_w, m_in, v_wn);

                // Increment iteration count LAST (like scalar)
                v_it = _mm512_mask_add_epi32(v_it, m_in, v_it, v_one);
            }

            uint32_t* dst = out + i * img_size + j;
            // Use aligned store if possible (buffer is 64-byte aligned and j is multiple of 16)
            if ((img_size % 16 == 0) && (reinterpret_cast<uintptr_t>(out) % 64 == 0)) {
                _mm512_mask_store_epi32((void*)dst, lane_mask, v_it);
            } else {
                _mm512_mask_storeu_epi32((void*)dst, lane_mask, v_it);
            }
        }
    }
}

// Helper function to save image as PPM
void save_ppm(const char* filename, uint32_t *data, uint32_t width, uint32_t height, uint32_t max_iters) {
    std::ofstream file(filename);
    file << "P3\n" << width << " " << height << "\n255\n";

    for (uint32_t i = 0; i < height; i++) {
        for (uint32_t j = 0; j < width; j++) {
            uint32_t val = data[i * width + j];
            // Simple color mapping
            if (val == max_iters) {
                file << "0 0 0 ";  // Black for points in the set
            } else {
                uint8_t r = (val * 9) % 256;
                uint8_t g = (val * 7) % 256;
                uint8_t b = (val * 5) % 256;
                file << (int)r << " " << (int)g << " " << (int)b << " ";
            }
        }
        file << "\n";
    }
    file.close();
}

// Verify two results match
bool verify_results(uint32_t *a, uint32_t *b, uint32_t size) {
    for (uint32_t i = 0; i < size * size; i++) {
        if (a[i] != b[i]) {
            std::cerr << "Mismatch at index " << i << ": " << a[i] << " vs " << b[i] << std::endl;
            return false;
        }
    }
    return true;
}

int main(int argc, char *argv[]) {
    uint32_t max_iters = 256;

    // Test sizes
    std::vector<uint32_t> test_sizes = {256, 512, 1024};

    std::cout << "Mandelbrot Benchmark\n";
    std::cout << "Testing sizes: 256x256, 512x512, 1024x1024\n";
    std::cout << "Max iterations: " << max_iters << "\n";
    std::cout << "==========================================\n\n";

    for (uint32_t img_size : test_sizes) {
        std::cout << "\n========================================\n";
        std::cout << "Image size: " << img_size << "x" << img_size << "\n";
        std::cout << "========================================\n";

        // Allocate output buffers (64-byte aligned for vector version)
        std::vector<uint32_t> out_scalar(img_size * img_size);
        aligned_vector<uint32_t> out_vector(img_size * img_size);

        // Test scalar version
        std::cout << "Running CPU scalar version...\n";
        auto start = std::chrono::high_resolution_clock::now();
        mandelbrot_cpu_scalar(img_size, max_iters, out_scalar.data());
        auto end = std::chrono::high_resolution_clock::now();
        auto scalar_time = std::chrono::duration<double, std::milli>(end - start).count();
        std::cout << "CPU Scalar time: " << scalar_time << " ms\n";

        // Save scalar output (only for 512x512 to avoid too many files)
        if (img_size == 512) {
            save_ppm("mandelbrot_scalar.ppm", out_scalar.data(), img_size, img_size, max_iters);
            std::cout << "Saved output to mandelbrot_scalar.ppm\n";
        }

        // Test vector version
        std::cout << "Running CPU vector version...\n";
        start = std::chrono::high_resolution_clock::now();
        mandelbrot_cpu_vector(img_size, max_iters, out_vector.data());
        end = std::chrono::high_resolution_clock::now();
        auto vector_time = std::chrono::duration<double, std::milli>(end - start).count();
        std::cout << "CPU Vector time: " << vector_time << " ms\n";

        // Verify results match
        if (verify_results(out_scalar.data(), out_vector.data(), img_size)) {
            std::cout << "✓ Vector results match scalar results\n";
        } else {
            std::cout << "✗ Vector results do NOT match scalar results\n";
        }

        std::cout << "Speedup: " << (scalar_time / vector_time) << "x\n";

        // Save vector output (only for 512x512)
        if (img_size == 512) {
            save_ppm("mandelbrot_vector.ppm", out_vector.data(), img_size, img_size, max_iters);
            std::cout << "Saved output to mandelbrot_vector.ppm\n";
        }
    }

    std::cout << "\n==========================================\n";
    std::cout << "All tests completed!\n";
    std::cout << "==========================================\n";

    return 0;
}


Overwriting mandelbrot_cpu.cpp


In [3]:
!g++ -O3 -march=x86-64-v4 -std=c++17 mandelbrot_cpu.cpp -o mandelbrot_cpu

In [4]:
!./mandelbrot_cpu

Mandelbrot Benchmark
Testing sizes: 256x256, 512x512, 1024x1024
Max iterations: 256


Image size: 256x256
Running CPU scalar version...
CPU Scalar time: 29.1412 ms
Running CPU vector version...
CPU Vector time: 2.9441 ms
✓ Vector results match scalar results
Speedup: 9.89816x

Image size: 512x512
Running CPU scalar version...
CPU Scalar time: 112.188 ms
Saved output to mandelbrot_scalar.ppm
Running CPU vector version...
CPU Vector time: 10.8191 ms
✓ Vector results match scalar results
Speedup: 10.3694x
Saved output to mandelbrot_vector.ppm

Image size: 1024x1024
Running CPU scalar version...
CPU Scalar time: 466.586 ms
Running CPU vector version...
CPU Vector time: 41.2672 ms
✓ Vector results match scalar results
Speedup: 11.3065x

All tests completed!


In [12]:
%%writefile mandelbrot_gpu.cu
#include <cstdint>
#include <cuda_runtime.h>
#include <iostream>
#include <vector>
#include <cstdlib>

#define CHECK_CUDA(call)                                                         \
    do {                                                                         \
        cudaError_t err__ = (call);                                              \
        if (err__ != cudaSuccess) {                                              \
            std::cerr << "CUDA error: " << cudaGetErrorString(err__)             \
                      << " at " << __FILE__ << ":" << __LINE__ << std::endl;     \
            std::exit(1);                                                        \
        }                                                                        \
    } while (0)

__global__ void mandelbrot_gpu_scalar(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out /* pointer to GPU memory */
) {
    for (uint32_t i = 0; i < img_size; i++) {
        for (uint32_t j = 0; j < img_size; j++) {
            float cx = (float(j) / float(img_size)) * 2.5f - 2.0f;
            float cy = (float(i) / float(img_size)) * 2.5f - 1.25f;

            float x2 = 0.0f;
            float y2 = 0.0f;
            float w = 0.0f;
            uint32_t iters = 0;
            while (x2 + y2 <= 4.0f && iters < max_iters) {
                float x = x2 - y2 + cx;
                float y = w - x2 - y2 + cy;
                x2 = x * x;
                y2 = y * y;
                w = (x + y) * (x + y);
                ++iters;
            }
            out[i * img_size + j] = iters;
        }
    }
}

void launch_mandelbrot_gpu_scalar(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out /* pointer to GPU memory */
) {
    mandelbrot_gpu_scalar<<<1, 1>>>(img_size, max_iters, out);
}

__global__ void mandelbrot_gpu_vector(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out /* pointer to GPU memory */
) {
    // Each thread processes one pixel position in a row.
    // threadIdx.x is the thread index inside the block (0-31).
    uint32_t tid = threadIdx.x;

    // Iterate through all rows.
    for (uint32_t i = 0; i < img_size; i++) {
        float cy = (float(i) / float(img_size)) * 2.5f - 1.25f;
        // thread 0 handles j=0, 32, 64, ...
        // thread 1 handles j=1, 33, 65, ...
        // and so on.
        for (uint32_t j = tid; j < img_size; j += 32) {
            float cx = (float(j) / float(img_size)) * 2.5f - 2.0f;

            float x2 = 0.0f;
            float y2 = 0.0f;
            float w = 0.0f;
            uint32_t iters = 0;

            while (x2 + y2 <= 4.0f && iters < max_iters) {
                float x = x2 - y2 + cx;
                float y = w - x2 - y2 + cy;
                x2 = x * x;
                y2 = y * y;
                w = (x + y) * (x + y);
                ++iters;
            }

            out[i * img_size + j] = iters;
        }
    }
}

void launch_mandelbrot_gpu_vector(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out /* pointer to GPU memory */
) {
    // Launch 1 block with 32 threads.
    // The 32 threads execute in one warp (32-wide SIMD-style execution).
    mandelbrot_gpu_vector<<<1, 32>>>(img_size, max_iters, out);
}


Overwriting mandelbrot_gpu.cu


In [14]:
%%writefile mandelbrot_gpu_parallel.cu
#include <cstdint>

// GPU Parallel Scalar: 每個 thread 處理一個像素
__global__ void mandelbrot_gpu_parallel_scalar(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out
) {
    // 2D grid 映射：每個 thread 對應一個像素
    uint32_t j = blockIdx.x * blockDim.x + threadIdx.x;  // 列 (x)
    uint32_t i = blockIdx.y * blockDim.y + threadIdx.y;  // 行 (y)

    // 邊界檢查
    if (i >= img_size || j >= img_size) return;

    // 計算該像素的複數坐標
    float cx = (float(j) / float(img_size)) * 2.5f - 2.0f;
    float cy = (float(i) / float(img_size)) * 2.5f - 1.25f;

    // Mandelbrot 迭代計算
    float x2 = 0.0f;
    float y2 = 0.0f;
    float w = 0.0f;
    uint32_t iters = 0;

    while (x2 + y2 <= 4.0f && iters < max_iters) {
        float x = x2 - y2 + cx;
        float y = w - x2 - y2 + cy;
        x2 = x * x;
        y2 = y * y;
        w = (x + y) * (x + y);
        ++iters;
    }

    // 寫入結果
    out[i * img_size + j] = iters;
}

void launch_mandelbrot_gpu_parallel_scalar(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out
) {
    // 使用 16x16 的 block（256 threads per block）
    dim3 block(16, 16);

    // 計算需要的 grid 大小
    dim3 grid((img_size + block.x - 1) / block.x,
              (img_size + block.y - 1) / block.y);

    mandelbrot_gpu_parallel_scalar<<<grid, block>>>(img_size, max_iters, out);
}

// GPU Parallel Vector: 使用更大的 block 來提升並行度
__global__ void mandelbrot_gpu_parallel_vector(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out
) {
    // 2D grid 映射
    uint32_t j = blockIdx.x * blockDim.x + threadIdx.x;
    uint32_t i = blockIdx.y * blockDim.y + threadIdx.y;

    if (i >= img_size || j >= img_size) return;

    float cx = (float(j) / float(img_size)) * 2.5f - 2.0f;
    float cy = (float(i) / float(img_size)) * 2.5f - 1.25f;

    float x2 = 0.0f;
    float y2 = 0.0f;
    float w = 0.0f;
    uint32_t iters = 0;

    while (x2 + y2 <= 4.0f && iters < max_iters) {
        float x = x2 - y2 + cx;
        float y = w - x2 - y2 + cy;
        x2 = x * x;
        y2 = y * y;
        w = (x + y) * (x + y);
        ++iters;
    }

    out[i * img_size + j] = iters;
}

void launch_mandelbrot_gpu_parallel_vector(
    uint32_t img_size,
    uint32_t max_iters,
    uint32_t *out
) {
    // 使用 32x32 的 block（1024 threads per block，最大值）
    dim3 block(32, 32);

    // 計算需要的 grid 大小
    dim3 grid((img_size + block.x - 1) / block.x,
              (img_size + block.y - 1) / block.y);

    mandelbrot_gpu_parallel_vector<<<grid, block>>>(img_size, max_iters, out);
}


Writing mandelbrot_gpu_parallel.cu


In [15]:
%%writefile test_mandelbrot_gpu.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <chrono>
#include <cstdint>
#include <cstring>
#include <cuda_runtime.h>

// Forward declarations for GPU functions (from mandelbrot_gpu.cu)
void launch_mandelbrot_gpu_scalar(uint32_t img_size, uint32_t max_iters, uint32_t *out);
void launch_mandelbrot_gpu_vector(uint32_t img_size, uint32_t max_iters, uint32_t *out);

// Forward declarations for parallel GPU functions (from mandelbrot_gpu2.cu)
void launch_mandelbrot_gpu_parallel_scalar(uint32_t img_size, uint32_t max_iters, uint32_t *out);
void launch_mandelbrot_gpu_parallel_vector(uint32_t img_size, uint32_t max_iters, uint32_t *out);

// Helper function to save image as PPM
void save_ppm(const char* filename, uint32_t *data, uint32_t width, uint32_t height, uint32_t max_iters) {
    std::ofstream file(filename);
    file << "P3\n" << width << " " << height << "\n255\n";

    for (uint32_t i = 0; i < height; i++) {
        for (uint32_t j = 0; j < width; j++) {
            uint32_t val = data[i * width + j];
            // Simple color mapping
            if (val == max_iters) {
                file << "0 0 0 ";  // Black for points in the set
            } else {
                uint8_t r = (val * 9) % 256;
                uint8_t g = (val * 7) % 256;
                uint8_t b = (val * 5) % 256;
                file << (int)r << " " << (int)g << " " << (int)b << " ";
            }
        }
        file << "\n";
    }
    file.close();
}

// Verify two results match
bool verify_results(uint32_t *a, uint32_t *b, uint32_t size) {
    for (uint32_t i = 0; i < size * size; i++) {
        if (a[i] != b[i]) {
            std::cerr << "Mismatch at index " << i << ": " << a[i] << " vs " << b[i] << std::endl;
            return false;
        }
    }
    return true;
}

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__ << " - " \
                      << cudaGetErrorString(err) << std::endl; \
            exit(1); \
        } \
    } while(0)

int main(int argc, char *argv[]) {
    uint32_t max_iters = 256;

    // Test sizes
    std::vector<uint32_t> test_sizes = {256, 512, 1024};

    std::cout << "Mandelbrot GPU Benchmark\n";
    std::cout << "Testing sizes: 256x256, 512x512, 1024x1024\n";
    std::cout << "Max iterations: " << max_iters << "\n";
    std::cout << "==========================================\n\n";

    for (uint32_t img_size : test_sizes) {
        std::cout << "\n========================================\n";
        std::cout << "Image size: " << img_size << "x" << img_size << "\n";
        std::cout << "========================================\n";

        // Allocate output buffers
        std::vector<uint32_t> out_gpu_scalar(img_size * img_size);
        std::vector<uint32_t> out_gpu_vector(img_size * img_size);
        std::vector<uint32_t> out_gpu_parallel_scalar(img_size * img_size);
        std::vector<uint32_t> out_gpu_parallel_vector(img_size * img_size);

        // ========== GPU SCALAR ==========
        std::cout << "Running GPU scalar version...\n";

        // Allocate GPU memory
        uint32_t *d_out;
        size_t buffer_size = img_size * img_size * sizeof(uint32_t);
        CUDA_CHECK(cudaMalloc(&d_out, buffer_size));

        auto start = std::chrono::high_resolution_clock::now();
        launch_mandelbrot_gpu_scalar(img_size, max_iters, d_out);
        CUDA_CHECK(cudaDeviceSynchronize());
        auto end = std::chrono::high_resolution_clock::now();
        auto gpu_scalar_time = std::chrono::duration<double, std::milli>(end - start).count();

        // Copy results back to CPU
        CUDA_CHECK(cudaMemcpy(out_gpu_scalar.data(), d_out, buffer_size, cudaMemcpyDeviceToHost));

        std::cout << "GPU Scalar time: " << gpu_scalar_time << " ms\n";

        // Save output (only for 512x512 to avoid too many files)
        if (img_size == 512) {
            save_ppm("mandelbrot_gpu_scalar.ppm", out_gpu_scalar.data(), img_size, img_size, max_iters);
            std::cout << "Saved output to mandelbrot_gpu_scalar.ppm\n";
        }

        // ========== GPU VECTOR ==========
        std::cout << "Running GPU vector version...\n";

        // Clear GPU memory
        CUDA_CHECK(cudaMemset(d_out, 0, buffer_size));

        start = std::chrono::high_resolution_clock::now();
        launch_mandelbrot_gpu_vector(img_size, max_iters, d_out);
        CUDA_CHECK(cudaDeviceSynchronize());
        end = std::chrono::high_resolution_clock::now();
        auto gpu_vector_time = std::chrono::duration<double, std::milli>(end - start).count();

        // Copy results back to CPU
        CUDA_CHECK(cudaMemcpy(out_gpu_vector.data(), d_out, buffer_size, cudaMemcpyDeviceToHost));

        std::cout << "GPU Vector time: " << gpu_vector_time << " ms\n";
        if (verify_results(out_gpu_scalar.data(), out_gpu_vector.data(), img_size)) {
            std::cout << "✓ GPU Vector results match GPU Scalar\n";
        } else {
            std::cout << "✗ GPU Vector results do NOT match GPU Scalar\n";
        }
        std::cout << "Speedup: " << (gpu_scalar_time / gpu_vector_time) << "x\n";

        // Save vector output (only for 512x512)
        if (img_size == 512) {
            save_ppm("mandelbrot_gpu_vector.ppm", out_gpu_vector.data(), img_size, img_size, max_iters);
            std::cout << "Saved output to mandelbrot_gpu_vector.ppm\n";
        }

        // ========== GPU PARALLEL SCALAR ==========
        std::cout << "Running GPU parallel scalar version (16x16 blocks)...\n";

        // Clear GPU memory
        CUDA_CHECK(cudaMemset(d_out, 0, buffer_size));

        start = std::chrono::high_resolution_clock::now();
        launch_mandelbrot_gpu_parallel_scalar(img_size, max_iters, d_out);
        CUDA_CHECK(cudaDeviceSynchronize());
        end = std::chrono::high_resolution_clock::now();
        auto gpu_parallel_scalar_time = std::chrono::duration<double, std::milli>(end - start).count();

        // Copy results back to CPU
        CUDA_CHECK(cudaMemcpy(out_gpu_parallel_scalar.data(), d_out, buffer_size, cudaMemcpyDeviceToHost));

        std::cout << "GPU Parallel Scalar time: " << gpu_parallel_scalar_time << " ms\n";
        if (verify_results(out_gpu_scalar.data(), out_gpu_parallel_scalar.data(), img_size)) {
            std::cout << "✓ GPU Parallel Scalar results match GPU Scalar\n";
        } else {
            std::cout << "✗ GPU Parallel Scalar results do NOT match GPU Scalar\n";
        }
        std::cout << "Speedup vs GPU Scalar: " << (gpu_scalar_time / gpu_parallel_scalar_time) << "x\n";

        // ========== GPU PARALLEL VECTOR ==========
        std::cout << "Running GPU parallel vector version (32x32 blocks)...\n";

        // Clear GPU memory
        CUDA_CHECK(cudaMemset(d_out, 0, buffer_size));

        start = std::chrono::high_resolution_clock::now();
        launch_mandelbrot_gpu_parallel_vector(img_size, max_iters, d_out);
        CUDA_CHECK(cudaDeviceSynchronize());
        end = std::chrono::high_resolution_clock::now();
        auto gpu_parallel_vector_time = std::chrono::duration<double, std::milli>(end - start).count();

        // Copy results back to CPU
        CUDA_CHECK(cudaMemcpy(out_gpu_parallel_vector.data(), d_out, buffer_size, cudaMemcpyDeviceToHost));

        std::cout << "GPU Parallel Vector time: " << gpu_parallel_vector_time << " ms\n";
        if (verify_results(out_gpu_scalar.data(), out_gpu_parallel_vector.data(), img_size)) {
            std::cout << "✓ GPU Parallel Vector results match GPU Scalar\n";
        } else {
            std::cout << "✗ GPU Parallel Vector results do NOT match GPU Scalar\n";
        }
        std::cout << "Speedup vs GPU Scalar: " << (gpu_scalar_time / gpu_parallel_vector_time) << "x\n";
        std::cout << "Speedup vs GPU Parallel Scalar: " << (gpu_parallel_scalar_time / gpu_parallel_vector_time) << "x\n";

        // Cleanup
        CUDA_CHECK(cudaFree(d_out));
    }

    std::cout << "\n==========================================\n";
    std::cout << "All tests completed!\n";
    std::cout << "==========================================\n";

    return 0;
}


Overwriting test_mandelbrot_gpu.cpp


In [16]:
!nvcc -std=c++17 -O3 mandelbrot_gpu.cu mandelbrot_gpu_parallel.cu test_mandelbrot_gpu.cpp -o test_mandelbrot_gpu

In [17]:
!./test_mandelbrot_gpu

Mandelbrot GPU Benchmark
Testing sizes: 256x256, 512x512, 1024x1024
Max iterations: 256


Image size: 256x256
Running GPU scalar version...
GPU Scalar time: 251.626 ms
Running GPU vector version...
GPU Vector time: 10.729 ms
✓ GPU Vector results match GPU Scalar
Speedup: 23.4529x
Running GPU parallel scalar version (16x16 blocks)...
GPU Parallel Scalar time: 30.7839 ms
✓ GPU Parallel Scalar results match GPU Scalar
Speedup vs GPU Scalar: 8.17393x
Running GPU parallel vector version (32x32 blocks)...
GPU Parallel Vector time: 0.089322 ms
✓ GPU Parallel Vector results match GPU Scalar
Speedup vs GPU Scalar: 2817.06x
Speedup vs GPU Parallel Scalar: 344.64x

Image size: 512x512
Running GPU scalar version...
GPU Scalar time: 912.735 ms
Saved output to mandelbrot_gpu_scalar.ppm
Running GPU vector version...
GPU Vector time: 36.235 ms
✓ GPU Vector results match GPU Scalar
Speedup: 25.1893x
Saved output to mandelbrot_gpu_vector.ppm
Running GPU parallel scalar version (16x16 blocks)...
GPU Para